In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input

In [7]:
df = pd.read_csv('../data/TSLA.csv')

df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

data = df[['Close']]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

train_size = int(len(scaled_data) * 0.8)

train_data = scaled_data[:train_size]
test_data = scaled_data[train_size-60:]

In [ ]:
def create_sequences(data, seq_length=60):
    X, y = [], []
    
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i])
        y.append(data[i])
        
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data)
X_test, y_test = create_sequences(test_data)

In [4]:
model = Sequential()

model.add(Input(shape=(X_train.shape[1], 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0077
Epoch 2/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 2.9872e-04
Epoch 3/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 2.3833e-04
Epoch 4/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 2.2273e-04
Epoch 5/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 2.1309e-04
Epoch 6/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 2.0891e-04
Epoch 7/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.9681e-04
Epoch 8/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.9376e-04
Epoch 9/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.8517e-04
Epoch 10/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 1.8089e-04


In [6]:
model.save('../models/lstm_best.keras')